In [ ]:

!pip install opencv-python


EXTRACTING DATA
::::::::::::::::::::::::::::::
::::::::::::::::::::::::::::::

In [ ]:
#extract from dataset
import os

your_dataset = []
base_folder = "GTSRB/Train"  # path to your Train folder

for label_folder in os.listdir(base_folder):
    label_path = os.path.join(base_folder, label_folder)
    if os.path.isdir(label_path):
        for filename in os.listdir(label_path):
            if filename.endswith(".ppm") or filename.endswith(".png") or filename.endswith(".jpg"):
                img_path = os.path.join(label_path, filename)
                your_dataset.append((img_path, int(label_folder)))  # label is the folder name as integer




!!!!hog CNN

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from skimage.feature import hog
from skimage.color import rgb2gray
from PIL import Image
import numpy as np

# ---------- 1️⃣ Dataset: CNN + HOG -------------
class GTSRBHOGDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert('RGB')
        img_np = np.array(img)
        gray = rgb2gray(img_np)

        hog_feature = hog(
            gray,
            orientations=9,
            pixels_per_cell=(8, 8),
            cells_per_block=(2, 2),
            block_norm='L2-Hys'
        )
        hog_feature = torch.tensor(hog_feature, dtype=torch.float32)

        if self.transform:
            img_tensor = self.transform(img)
        else:
            img_tensor = transforms.ToTensor()(img)

        label = self.labels[idx]
        return img_tensor, hog_feature, label

# ---------- 2️⃣ Model: CNN + HOG Fusion -------------
class CNN_HOG_Model(nn.Module):
    def __init__(self, hog_dim, num_classes=43):
        super(CNN_HOG_Model, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)

        # Assuming input image 32x32 -> pool -> 16x16 -> pool -> 8x8
        self.cnn_out_dim = 64 * 8 * 8

        self.fc1 = nn.Linear(self.cnn_out_dim + hog_dim, 256)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x_img, x_hog):
        x = F.relu(self.conv1(x_img))
        x = self.pool(x)
        x = F.relu(self.conv2(x))
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        x = torch.cat((x, x_hog), dim=1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# ---------- 3️⃣ Dummy paths & labels (replace!) -------------
# This is placeholder: Replace with your real file paths and labels
# Load sample image paths from GTSRB/Train if available
import glob
gtsrb_imgs = glob.glob('GTSRB/Train/*/*.png') + glob.glob('GTSRB/Train/*/*.ppm')
if len(gtsrb_imgs) >= 2:
    train_image_paths = gtsrb_imgs[:10]
    train_labels = [0] * len(train_image_paths)
else:
    train_image_paths = []
    train_labels = []


# ---------- 4️⃣ Transforms -------------
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor()
])

# ---------- 5️⃣ Loaders -------------
train_dataset = GTSRBHOGDataset(train_image_paths, train_labels, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)

# ---------- 6️⃣ Initialize -------------
# Get HOG dim from one sample
hog_dim = len(hog(np.zeros((32, 32)), orientations=9, pixels_per_cell=(8, 8), cells_per_block=(2, 2)))
model = CNN_HOG_Model(hog_dim=hog_dim, num_classes=43)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

# ---------- 7️⃣ Training Loop -------------
for epoch in range(2):  # example epochs
    model.train()
    for images, hogs, labels in train_loader:
        images = images.to(device)
        hogs = hogs.to(device)
        labels = labels.to(device)

        outputs = model(images, hogs)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch [{epoch+1}], Loss: {loss.item():.4f}")

print("✅ Done")


In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from skimage.feature import hog
from skimage.color import rgb2gray
from PIL import Image
import numpy as np
from sklearn.model_selection import train_test_split

# ---------- ✅ 1️⃣ EXTRACT IMAGE PATHS + LABELS ----------
your_dataset = []
base_folder = "GTSRB/Train"  # update with your actual path

for label_folder in os.listdir(base_folder):
    label_path = os.path.join(base_folder, label_folder)
    if os.path.isdir(label_path):
        for filename in os.listdir(label_path):
            if filename.endswith(".ppm") or filename.endswith(".png") or filename.endswith(".jpg"):
                img_path = os.path.join(label_path, filename)
                your_dataset.append((img_path, int(label_folder)))

# Split into separate lists
image_paths = [x[0] for x in your_dataset]
labels = [x[1] for x in your_dataset]

# ---------- ✅ 2️⃣ TRAIN / VAL SPLIT ----------
train_paths, val_paths, train_labels, val_labels = train_test_split(
    image_paths, labels, test_size=0.2, stratify=labels, random_state=42
)

# ---------- ✅ 3️⃣ DATASET: HOG + IMAGE ----------
class GTSRBHOGDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert('RGB')
        img = img.resize((32, 32))
        img_np = np.array(img)
        gray = rgb2gray(img_np)

        hog_feature = hog(
            gray,
            orientations=9,
            pixels_per_cell=(8, 8),
            cells_per_block=(2, 2),
            block_norm='L2-Hys'
        )
        hog_feature = torch.tensor(hog_feature, dtype=torch.float32)

        if self.transform:
            img_tensor = self.transform(img)
        else:
            img_tensor = transforms.ToTensor()(img)

        label = self.labels[idx]
        return img_tensor, hog_feature, label

# ---------- ✅ 4️⃣ TRANSFORM ----------
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor()
])

# ---------- ✅ 5️⃣ DATALOADERS ----------
train_dataset = GTSRBHOGDataset(train_paths, train_labels, transform=transform)
val_dataset = GTSRBHOGDataset(val_paths, val_labels, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

# ---------- ✅ 6️⃣ MODEL: CNN + HOG Fusion ----------
class CNN_HOG_Model(nn.Module):
    def __init__(self, hog_dim, num_classes=43):
        super(CNN_HOG_Model, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.cnn_out_dim = 64 * 8 * 8  # for 32x32 input

        self.fc1 = nn.Linear(self.cnn_out_dim + hog_dim, 256)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x_img, x_hog):
        x = F.relu(self.conv1(x_img))
        x = self.pool(x)
        x = F.relu(self.conv2(x))
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        x = torch.cat((x, x_hog), dim=1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# ---------- ✅ 7️⃣ INIT ----------
hog_dim = len(hog(np.zeros((32, 32)), orientations=9, pixels_per_cell=(8, 8), cells_per_block=(2, 2)))
model = CNN_HOG_Model(hog_dim=hog_dim, num_classes=43)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

# ---------- ✅ 8️⃣ TRAIN LOOP ----------
for epoch in range(2):  # change to more epochs!
    model.train()
    for images, hogs, labels in train_loader:
        images = images.to(device)
        hogs = hogs.to(device)
        labels = labels.to(device)

        outputs = model(images, hogs)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch [{epoch+1}], Loss: {loss.item():.4f}")

print("✅ DONE with your GTSRB + CNN + HOG pipeline")


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# ---------- ✅ 1️⃣ Validation Loop -------------
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, hogs, labels in val_loader:
        images = images.to(device)
        hogs = hogs.to(device)
        labels = labels.to(device)

        outputs = model(images, hogs)
        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# ---------- ✅ 2️⃣ Compute Metrics -------------
accuracy = accuracy_score(all_labels, all_preds)
precision_macro = precision_score(all_labels, all_preds, average='macro')
recall_macro = recall_score(all_labels, all_preds, average='macro')
f1_macro = f1_score(all_labels, all_preds, average='macro')

precision_weighted = precision_score(all_labels, all_preds, average='weighted')
recall_weighted = recall_score(all_labels, all_preds, average='weighted')
f1_weighted = f1_score(all_labels, all_preds, average='weighted')

print(f"\nValidation Accuracy: {accuracy:.4f}")

print(f"Macro Precision: {precision_macro:.4f}")
print(f"Macro Recall: {recall_macro:.4f}")
print(f"Macro F1 Score: {f1_macro:.4f}")

print(f"Weighted Precision: {precision_weighted:.4f}")
print(f"Weighted Recall: {recall_weighted:.4f}")
print(f"Weighted F1 Score: {f1_weighted:.4f}")

print("\nDetailed Classification Report:")
print(classification_report(all_labels, all_preds))


!!!cnn hog without problematic class!!

In [ ]:
import os
import cv2
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Subset, DataLoader
from torchvision import datasets, transforms, models

from sklearn import svm
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# ---------------------------
# 1️⃣ CONFIG
# ---------------------------
DATA_DIR = 'GTSRB/Train' if os.path.exists('GTSRB/Train') else 'GTSRB/Train'  # your data folder
REMOVE_CLASSES = ['0', '19', '32', '37', '41']

# ---------------------------
# 2️⃣ CNN PART
# ---------------------------

# Transform for CNN
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor()
])

# Load all data first
full_dataset = datasets.ImageFolder(root="GTSRB/Train", transform=transform)

# Figure out which indices to remove
class_to_idx = full_dataset.class_to_idx
remove_indices = [class_to_idx[c] for c in REMOVE_CLASSES if c in class_to_idx]

# Keep only samples not in remove list
keep_indices = [i for i, (_, label) in enumerate(full_dataset.samples) if label not in remove_indices]

filtered_dataset = Subset(full_dataset, keep_indices)
train_loader = DataLoader(filtered_dataset, batch_size=32, shuffle=True)

# Simple CNN using ResNet18
model = models.resnet18(pretrained=False)
model.fc = nn.Linear(model.fc.in_features, len(full_dataset.classes) - len(remove_indices))

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print(f"CNN Training with {len(keep_indices)} images and {len(full_dataset.classes) - len(remove_indices)} classes")

# One quick training loop example
model.train()
for epoch in range(1):  # keep small for test
    total_loss = 0.0
    for images, labels in train_loader:
        outputs = model(images)
        # Remap labels to new indices
        new_labels = torch.tensor([list(sorted(set(range(len(full_dataset.classes))) - set(remove_indices))).index(l.item()) for l in labels])
        loss = criterion(outputs, new_labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch [{epoch+1}] Loss: {total_loss:.4f}")

# ---------------------------
# 3️⃣ HOG + SVM PART
# ---------------------------

print("\nHOG + SVM training...")

hog = cv2.HOGDescriptor()
X, y = [], []

# Get valid class folders
class_folders = [f for f in os.listdir(DATA_DIR) if f not in REMOVE_CLASSES and os.path.isdir(os.path.join(DATA_DIR, f))]

print(f"Using classes: {class_folders}")

# Load images + compute HOG features
for class_idx, class_name in enumerate(sorted(class_folders)):
    class_dir = os.path.join(DATA_DIR, class_name)
    for img_name in os.listdir(class_dir):
        img_path = os.path.join(class_dir, img_name)
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            continue
        img = cv2.resize(img, (64, 128))
        features = hog.compute(img).flatten()
        X.append(features)
        y.append(class_idx)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

clf = svm.SVC()
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"HOG + SVM Accuracy: {acc:.4f}")
